# OAB Pipeline — Benchmark de Modelos de Linguagem

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/diegofnf/Topicos_Avancados_2026-1_Equipe_JUD_4_atividade1/blob/main/diego_bispo/notebook.ipynb)

**Pipeline:**
1. Setup
2. Preparo dos dados
3. Download dos modelos
4. Respostas dos candidatos
5. Curadoria
6. Avaliação discursiva
7. Análise quantitativa
8. Análise qualitativa
9. Análise qualitativa Reference
10. Análise qualitativa Reference-Free
11. Resultados
12. Resumo e push para o Git


## 1. Setup

In [1]:
import os
import shutil

# Baixa apenas o conteúdo de diego_bispo para a pasta de trabalho e remove o checkout temporário.
if not os.path.exists('/content/oab_pipeline'):
    !git clone --filter=blob:none --no-checkout https://github.com/diegofnf/Topicos_Avancados_2026-1_Equipe_JUD_4_atividade1 /content/oab_repo_tmp
    %cd /content/oab_repo_tmp
    !git sparse-checkout init --cone
    !git sparse-checkout set diego_bispo
    !git checkout main

    shutil.copytree('/content/oab_repo_tmp/diego_bispo', '/content/oab_pipeline')
    shutil.rmtree('/content/oab_repo_tmp')

%cd /content/oab_pipeline

Cloning into '/content/oab_repo_tmp'...
remote: Enumerating objects: 70, done.
remote: Counting objects: 100% (70/70), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 70 (delta 23), reused 57 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (70/70), 9.99 KiB | 601.00 KiB/s, done.
Resolving deltas: 100% (23/23), done.
/content/oab_repo_tmp
remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 19 (delta 4), reused 10 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (19/19), 22.11 KiB | 3.68 MiB/s, done.
Resolving deltas: 100% (4/4), done.
Already on 'main'
Your branch is up to date with 'origin/main'.
/content/oab_pipeline


In [ ]:
!pip install -q transformers accelerate bitsandbytes datasets pandas tqdm scikit-learn textstat bert-score sentence-transformers seaborn matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.1 MB/s eta 0:00:00


In [3]:
import numpy as np
import torch
import pandas as pd
from google.colab import userdata
from huggingface_hub import snapshot_download

from config import (
    MODELOS_CANDIDATOS, MODELO_CURADOR, MODELO_JUIZ,
    QUESTOES_DISCURSIVAS_CSV, QUESTOES_OBJETIVAS_CSV,
    CURADORIA_DISCURSIVAS_CSV, CURADORIA_OBJETIVAS_CSV,
    RESPOSTAS_DISCURSIVAS_CSV, RESPOSTAS_OBJETIVAS_CSV,
    AVALIACAO_DISCURSIVAS_CSV, AVALIACAO_DISCURSIVAS_REFERENCE_CSV,
    AVALIACAO_DISCURSIVAS_REFERENCE_FREE_CSV, SIMILARIDADE_DISCURSIVAS_CSV,
    HEATMAP_DISCURSIVAS_PNG, BENCHMARK_OBJETIVAS_CSV,
    BENCHMARK_DISCURSIVAS_CSV, BENCHMARK_DISCURSIVAS_REFERENCE_CSV,
    BENCHMARK_DISCURSIVAS_REFERENCE_FREE_CSV,
    DISC_SLICE_START, DISC_SLICE_END,
    OBJ_SLICE_START, OBJ_SLICE_END,
    REPO_DIR, GITHUB_REPO,
)

# Reprodutibilidade
# Seed não remove a aleatoriedade — ele só faz ela se comportar sempre igual.
# 42 vem do livro O Guia do Mochileiro das Galáxias (de Douglas Adams) - a resposta para a vida é to be
np.random.seed(42)
torch.manual_seed(42)

# Autenticação Hugging Face
token = userdata.get('HF_TOKEN')
if not token:
    raise ValueError('HF_TOKEN não encontrado. Configure nas Secrets do Colab.')
os.environ['HF_TOKEN'] = token
# GPU obrigatória
if not torch.cuda.is_available():
    raise RuntimeError('GPU não disponível. Ative em Ambiente de execução > Alterar tipo de hardware > T4.')
print(f'GPU: {torch.cuda.get_device_name(0)}')

{'type': 'user', 'id': '69a9db075a0c5c017e67d1ca', 'name': 'diegobispo', 'fullname': 'Diego Bispo', 'email': 'diego.fnf@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1775001600, 'isPro': False, 'avatarUrl': '/avatars/d22facc0764a4f721bde1c808e0899de.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'Atividade1', 'role': 'read', 'createdAt': '2026-03-15T01:20:08.430Z'}}}


RuntimeError: GPU não disponível. Ative em Ambiente de execução > Alterar tipo de hardware > T4.

## 2. Preparo dos dados

In [ ]:
from data_utils import carregar_csv
from preparar_dados import preparar_dados

df_disc, df_obj = preparar_dados()
df_cur_disc = carregar_csv(CURADORIA_DISCURSIVAS_CSV)
df_cur_obj = carregar_csv(CURADORIA_OBJETIVAS_CSV)

display(df_disc.head())
display(df_obj.head())
display(df_cur_disc.head())
display(df_cur_obj.head())


## 3. Download dos modelos

In [ ]:
from config import HF_CACHE_DIR
# O set() elimina duplicatas — MODELO_CURADOR e MODELO_JUIZ são o mesmo modelo,
# sem o set() o download seria feito duas vezes
print('Baixando modelos uma única vez...')
for model_name in set(list(MODELOS_CANDIDATOS.values()) + [MODELO_CURADOR, MODELO_JUIZ]):
    print(f'Baixando: {model_name}')
    snapshot_download(repo_id=model_name, cache_dir=HF_CACHE_DIR)

## 4. Respostas dos candidatos

In [ ]:
from tqdm import tqdm
from data_utils import carregar_csv, salvar_csv
from model_utils import load_model, unload
from pipeline import gerar_resposta_discursiva, gerar_resposta_objetiva

questoes_disc = carregar_csv(QUESTOES_DISCURSIVAS_CSV)
questoes_obj  = carregar_csv(QUESTOES_OBJETIVAS_CSV)

respostas_discursivas = []
resultados_obj = []

for nome_modelo, path_modelo in MODELOS_CANDIDATOS.items():
    print(f'Processando candidato: {nome_modelo}')
    candidato_model, candidato_tok = load_model(path_modelo)

    print('  → Discursivas...')
    for _, row in tqdm(questoes_disc.iterrows(), total=len(questoes_disc)):
        respostas_discursivas.append(
            gerar_resposta_discursiva(candidato_model, candidato_tok, row.to_dict(), nome_modelo)
        )

    print('  → Objetivas...')
    for _, row in tqdm(questoes_obj.iterrows(), total=len(questoes_obj)):
        resultados_obj.append(
            gerar_resposta_objetiva(candidato_model, candidato_tok, row.to_dict(), nome_modelo)
        )

    unload(candidato_model, candidato_tok)

df_resp_disc = pd.DataFrame(respostas_discursivas)
df_obj_res   = pd.DataFrame(resultados_obj)

salvar_csv(df_resp_disc, RESPOSTAS_DISCURSIVAS_CSV)
salvar_csv(df_obj_res,   RESPOSTAS_OBJETIVAS_CSV)

display(df_resp_disc.head())
display(df_obj_res.head())

## 5. Curadoria

In [ ]:
from data_utils import carregar_csv

print('Curadoria por modelo desabilitada: usando curadoria externa versionada no reposit?rio.')
df_cur_disc = carregar_csv(CURADORIA_DISCURSIVAS_CSV)
df_cur_obj = carregar_csv(CURADORIA_OBJETIVAS_CSV)

display(df_cur_disc.head())
display(df_cur_obj.head())


## 6. Avaliação discursiva

In [ ]:
from tqdm import tqdm
from data_utils import carregar_csv, salvar_csv
from model_utils import load_model, unload
from pipeline import gerar_avaliacao_discursiva

questoes_disc  = carregar_csv(QUESTOES_DISCURSIVAS_CSV)
respostas_disc = carregar_csv(RESPOSTAS_DISCURSIVAS_CSV)
curadoria_disc = carregar_csv(CURADORIA_DISCURSIVAS_CSV)

juiz_model, juiz_tok = load_model(MODELO_JUIZ)
avaliacoes_discursivas = []

for _, resposta_row in tqdm(respostas_disc.iterrows(), total=len(respostas_disc)):
    resposta_row  = resposta_row.to_dict()
    questao_row   = questoes_disc.loc[questoes_disc['question_id'] == resposta_row['question_id']].iloc[0].to_dict()
    curadoria_row = curadoria_disc.loc[curadoria_disc['question_id'] == resposta_row['question_id']].iloc[0].to_dict()
    avaliacoes_discursivas.append(
        gerar_avaliacao_discursiva(juiz_model, juiz_tok, questao_row, resposta_row, curadoria_row)
    )

unload(juiz_model, juiz_tok)

df_avaliacao_disc = pd.DataFrame(avaliacoes_discursivas)
salvar_csv(df_avaliacao_disc, AVALIACAO_DISCURSIVAS_CSV)
display(df_avaliacao_disc.head())

### Setup mínimo para reexecutar apenas a análise após reiniciar a sessão

Se a sessão do Colab for reiniciada e os CSVs já estiverem no repositório, execute a célula abaixo antes das seções 7, 8, 9 e 10.


In [ ]:
import os
import sys
import shutil

if not os.path.exists('/content/oab_pipeline'):
    !git clone --filter=blob:none --no-checkout https://github.com/diegofnf/Topicos_Avancados_2026-1_Equipe_JUD_4_atividade1 /content/oab_repo_tmp
    %cd /content/oab_repo_tmp
    !git sparse-checkout init --cone
    !git sparse-checkout set diego_bispo
    !git checkout main
    shutil.copytree('/content/oab_repo_tmp/diego_bispo', '/content/oab_pipeline')
    shutil.rmtree('/content/oab_repo_tmp')

%cd /content/oab_pipeline
!pip install -q -r requirements.txt

if '/content/oab_pipeline' not in sys.path:
    sys.path.append('/content/oab_pipeline')


## 7. Análise quantitativa

In [ ]:
import pandas as pd

from analise_resultados import executar_analise

resultados_analise = executar_analise()

display(pd.DataFrame([
    {
        'acuracia_global': resultados_analise['acuracia_global'],
    }
]).round(4))

display(resultados_analise['benchmark_objetivas'].round(4))


## 8. Análise qualitativa

In [ ]:
from IPython.display import Image
from config import HEATMAP_DISCURSIVAS_PNG

resultados_analise = executar_analise(incluir_qualitativa=True)

benchmark_qualitativo = resultados_analise['benchmark_final'][[
    'modelo',
    'nota_media',
]].round(4)

display(benchmark_qualitativo)
display(resultados_analise['bertscore_discursivas'].round(4))
display(Image(filename=str(HEATMAP_DISCURSIVAS_PNG)))


## 9. Análise qualitativa Reference


In [ ]:
from config import (
    AVALIACAO_DISCURSIVAS_REFERENCE_CSV,
    BENCHMARK_DISCURSIVAS_REFERENCE_CSV,
    QUESTOES_DISCURSIVAS_CSV,
    RESPOSTAS_DISCURSIVAS_CSV,
)
from data_utils import carregar_csv, salvar_csv
from metricas_qualitativas_reference import evaluate_dataframe

df_respostas_reference = carregar_csv(RESPOSTAS_DISCURSIVAS_CSV).merge(
    carregar_csv(QUESTOES_DISCURSIVAS_CSV)[['question_id', 'gabarito_narrativo', 'gabarito_itens_json']],
    on='question_id',
    how='left',
)

df_ref_detalhe, df_ref_resumo = evaluate_dataframe(df_respostas_reference)

salvar_csv(df_ref_detalhe, AVALIACAO_DISCURSIVAS_REFERENCE_CSV)
salvar_csv(df_ref_resumo, BENCHMARK_DISCURSIVAS_REFERENCE_CSV)

display(df_ref_resumo.round(4))
display(df_ref_detalhe.sort_values(['question_id', 'ranking']).round(4))


## 10. Análise qualitativa Reference-Free


In [ ]:
from config import (
    AVALIACAO_DISCURSIVAS_REFERENCE_FREE_CSV,
    BENCHMARK_DISCURSIVAS_REFERENCE_FREE_CSV,
    RESPOSTAS_DISCURSIVAS_CSV,
)
from data_utils import carregar_csv, salvar_csv
from metricas_qualitativas_reference_free import evaluate_dataframe

df_ref_free_detalhe, df_ref_free_resumo = evaluate_dataframe(
    carregar_csv(RESPOSTAS_DISCURSIVAS_CSV)
)

salvar_csv(df_ref_free_detalhe, AVALIACAO_DISCURSIVAS_REFERENCE_FREE_CSV)
salvar_csv(df_ref_free_resumo, BENCHMARK_DISCURSIVAS_REFERENCE_FREE_CSV)

display(df_ref_free_resumo.round(4))
display(df_ref_free_detalhe.sort_values(['question_id', 'ranking']).round(4))


## 11. Resultados


In [ ]:
display(resultados_analise['benchmark_final'].round(4))


## 12. Resumo e push para o Git


In [5]:
import shutil
from pathlib import Path
from data_utils import carregar_csv, git_push

csvs = {
    'questoes_discursivas':  QUESTOES_DISCURSIVAS_CSV,
    'questoes_objetivas':    QUESTOES_OBJETIVAS_CSV,
    'curadoria_discursivas': CURADORIA_DISCURSIVAS_CSV,
    'curadoria_objetivas':   CURADORIA_OBJETIVAS_CSV,
    'respostas_discursivas': RESPOSTAS_DISCURSIVAS_CSV,
    'respostas_objetivas':   RESPOSTAS_OBJETIVAS_CSV,
    'avaliacao_discursivas': AVALIACAO_DISCURSIVAS_CSV,
    'avaliacao_discursiva_reference': AVALIACAO_DISCURSIVAS_REFERENCE_CSV,
    'benchmark_discursivas_reference': BENCHMARK_DISCURSIVAS_REFERENCE_CSV,
    'avaliacao_discursiva_reference_free': AVALIACAO_DISCURSIVAS_REFERENCE_FREE_CSV,
    'benchmark_discursivas_reference_free': BENCHMARK_DISCURSIVAS_REFERENCE_FREE_CSV,
    'similaridade_discursivas': SIMILARIDADE_DISCURSIVAS_CSV,
    'benchmark_objetivas':   BENCHMARK_OBJETIVAS_CSV,
    'benchmark_discursivas': BENCHMARK_DISCURSIVAS_CSV,
    'heatmap_discursivas':   HEATMAP_DISCURSIVAS_PNG,
}

print('Resumo dos CSVs gerados:')
for nome, path in csvs.items():
    if path.suffix == '.csv':
        print(f'  {nome}: {len(carregar_csv(path))} linhas')
    else:
        status_arquivo = 'gerado' if path.exists() else 'não gerado'
        print(f'  {nome}: {status_arquivo}')

# Push - persiste os resultados no repositório Git
# Configure GITHUB_TOKEN nas Secrets do Colab (mesmo local do HF_TOKEN)
github_token = userdata.get('GITHUB_TOKEN')
if github_token:
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)

    !git clone --filter=blob:none --no-checkout https://github.com/diegofnf/Topicos_Avancados_2026-1_Equipe_JUD_4_atividade1 {REPO_DIR}
    %cd {REPO_DIR}
    !git sparse-checkout init --cone
    !git sparse-checkout set diego_bispo
    !git checkout main

    destino_repo = Path('/content/oab_repo/diego_bispo')
    if destino_repo.exists():
        shutil.rmtree(destino_repo)
    shutil.copytree('/content/oab_pipeline', destino_repo)

    git_push(REPO_DIR, GITHUB_REPO, github_token)
    shutil.rmtree(REPO_DIR)
else:
    print('GITHUB_TOKEN não encontrado nas Secrets. CSVs salvos apenas localmente.')

print('\nPipeline finalizado com sucesso.')


Resumo dos CSVs gerados:
  questoes_discursivas: 1 linhas
  questoes_objetivas: 1 linhas
  curadoria_discursivas: 1 linhas
  curadoria_objetivas: 1 linhas
  respostas_discursivas: 3 linhas
  respostas_objetivas: 3 linhas
  avaliacao_discursivas: 3 linhas
  benchmark_objetivas: 3 linhas
  benchmark_discursivas: 2 linhas
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
Cloning into '/content/oab_repo'...
fatal: Unable to read current working directory: No such file or directory
[Errno 2] No such file or directory: '/content/oab_repo'
/content/oab_repo
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or direc